# Pipeline - Treino / Previsão

In [44]:
from utils.common import import_dataframe
from utils.common import make_lags, make_leads
from utils.common import calculate_metrics
from utils.common import plot_plotly
from utils.common import ModeloPrevisaoVolume

import pandas as pd
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

## Importar dados

In [ ]:
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
nome_coluna_vazao_jusante = "Vazão Jusante (m³/s)"

df = import_dataframe()
df_volume = df[["Data", nome_coluna_volume]].copy()
df_vazao_natural = df[["Data", nome_coluna_vazao_natural]].copy()
df_vazao_jusante = df[["Data", nome_coluna_vazao_jusante]].copy()

df_volume = df_volume.set_index("Data")
df_vazao_natural = df_vazao_natural.set_index("Data")
df_vazao_jusante = df_vazao_jusante.set_index("Data")

df_volume_series = (
  df_volume
    .groupby('Data').mean()
    .squeeze()
)

df_vazao_natural_series = (
  df_vazao_natural
    .groupby('Data').mean()
    .squeeze()
)

df_vazao_jusante_series = (
  df_vazao_jusante
    .groupby('Data').mean()
    .squeeze()
)

## Preprocessar dados

### Vazão de entrada

In [ ]:
y_vn = df_vazao_natural_series[df_vazao_natural_series.index >= "2018-01-01"].copy()

fourier = CalendarFourier(freq="Y", order=2)
dp = DeterministicProcess(
  index=y_vn.index,
  constant=False,
  order=1,
  seasonal=False,
  additional_terms=[fourier],
  drop=True,
)
X_full_vn = dp.in_sample()

VALIDATION_SIZE = 1*90

X_vn_train_rec, X_vn_valid_rec, y_vn_train_rec, y_vn_valid_rec = train_test_split(X_full_vn, y_vn, test_size=VALIDATION_SIZE, shuffle=False)

### Vazão jusante

In [47]:
y_vj = df_vazao_jusante_series[df_vazao_jusante_series.index >= "2018-01-01"].copy()

fourier = CalendarFourier(freq="Y", order=2)
dp = DeterministicProcess(
  index=y_vj.index,
  constant=False,
  order=1,
  seasonal=False,
  additional_terms=[fourier],
  drop=True,
)
X_full_vj = dp.in_sample()

X_vj_train_rec, X_vj_valid_rec, y_vj_train_rec, y_vj_valid_rec = train_test_split(X_full_vj, y_vj, test_size=VALIDATION_SIZE, shuffle=False)

### Volume

In [48]:
def get_train_test_split_recursive():
  y_vol = df_volume_series[df_volume_series.index >= "2018-01-01"].copy()
  X_lags = make_lags(y_vol.squeeze(), 1)
  X_Qin_leads = make_leads(y_vn.squeeze(), 1, name="Qin")
  X_Qout_leads = make_leads(y_vj.squeeze(), 1, name="Qout")
  X_full_vol = pd.concat([X_lags, X_Qin_leads, X_Qout_leads], axis=1).dropna()

  y_vol, X_full_vol = y_vol.align(X_full_vol, join='inner', axis=0)

  X_vol_train_rec, X_vol_valid_rec, y_vol_train_rec, y_vol_valid_rec = train_test_split(X_full_vol, y_vol, test_size=VALIDATION_SIZE, shuffle=False)
  return y_vol, X_full_vol, X_vol_train_rec, X_vol_valid_rec, y_vol_train_rec, y_vol_valid_rec

y_vol_rec, X_full_vol, X_vol_train_rec, X_vol_valid_rec, y_vol_train_rec, y_vol_valid_rec = get_train_test_split_recursive()

## Modelagem para volume desacoplando o modelo da recursão

In [49]:
model = ModeloPrevisaoVolume(LinearRegression(fit_intercept=False))

model.fit_vazao_natural(X_vn_train_rec, y_vn_train_rec)
model.fit_vazao_jusante(X_vj_train_rec, y_vj_train_rec)
model.calculate_lags(X_vn_valid_rec)
model.fit(X_vol_train_rec, y_vol_train_rec)

X_vn_train_rec_drop = X_vn_train_rec.drop(pd.Timestamp("2018-01-01"))
y_fit = model.predict(X_vn_train_rec_drop)
y_pred = model.predict(X_vn_valid_rec)

metric = calculate_metrics(
  y_vol_rec.loc[y_fit.index],
  y_vol_rec.loc[y_pred.index],
  y_fit.squeeze(),
  y_pred.squeeze()
)
display(metric)

fig1 = plot_plotly(
  y_vol_rec.loc[y_fit.index],
  y_fit,
  title="Forecast - Treino",
  showlegend=True
)

fig2 = plot_plotly(
  y_vol_rec.loc[y_pred.index],
  y_pred,
  title="Forecast - Validação",
  showlegend=True
)

c:\Users\marce\Desktop\Pos Graduacoes\UFSCAR - MachineLearningInProduction\AtividadesEntregas\4_TCC\codigo\tcc\.venv\lib\site-packages\sklearn\base.py:1365: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().



,RMSE,MSE,MAE,R2,Wasserstein
Treino,0.037984,0.001443,0.031208,0.999994,0.022650
Validação,16.937709,286.885972,14.490483,-4.393054,14.490454


16418255


76510071


## Previsão para o futuro

In [50]:
model.update(X_full_vol, y_vol_rec, y_vn, y_vj)

In [51]:
fourier = CalendarFourier(freq="Y", order=2)
y_vn = y_vn.asfreq("D")
dp = DeterministicProcess(
  index=y_vn.index,
  constant=False,
  order=1,
  seasonal=False,
  additional_terms=[fourier],
  drop=True,
)
X_full_vn = dp.in_sample()

X_future_vn = dp.out_of_sample(steps=90)

In [52]:
model.calculate_lags(X_future_vn)
y_pred = model.predict(X_future_vn)

fig2 = plot_plotly(
  y_vol_rec.loc[y_vol_rec.index],
  y_pred,
  title="Forecast - Validação",
  showlegend=True
)

89232901
